<a href="https://colab.research.google.com/github/fataa34/applied-optimization-with-python/blob/constraint/Constraint_(Direct).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import linprog, minimize

In [ ]:
def objective_function(x):
  return x[0]**4-2*x[0]**2*x[1]+x[0]**2+x[0]*x[1]**2-2*x[0]+4

def eq1(x):
  return x[0]**2+x[1]**2-2

def ineq1(x):
  return 0.25*x[0]**2+0.75*x[1]**2-1

# SLP

In [ ]:
def sequential_linear_programming(target_func, x0, eq_constraints, ineq_constraints, Ns, epsilon, move_limit):

    def get_gradient(func, x):
        h_step = 1e-5
        n = len(x)
        grad = np.zeros(n)
        f_x = func(x)
        for i in range(n):
            x_tmp = x.copy()
            x_tmp[i] += h_step
            grad[i] = (func(x_tmp) - f_x) / h_step
        return grad

    x_current = np.array(x0, dtype=float)
    n_vars = len(x_current)

    f_val = target_func(x_current)
    val_eq_init = [h(x_current) for h in eq_constraints]
    val_ineq_init = [g(x_current) for g in ineq_constraints]

    history_data = []
    history_data.append([0, x_current.copy(), f_val, val_eq_init, val_ineq_init])

    for q in range(1, Ns + 1):
        c = get_gradient(target_func, x_current)

        A_ub = []
        b_ub = []
        for g_func in ineq_constraints:
            grad_g = get_gradient(g_func, x_current)
            val_g = g_func(x_current)
            A_ub.append(grad_g)
            b_ub.append(-val_g)

        A_eq = []
        b_eq = []
        for h_func in eq_constraints:
            grad_h = get_gradient(h_func, x_current)
            val_h = h_func(x_current)
            A_eq.append(grad_h)
            b_eq.append(-val_h)

        A_ub = np.array(A_ub) if A_ub else None
        b_ub = np.array(b_ub) if b_ub else None
        A_eq = np.array(A_eq) if A_eq else None
        b_eq = np.array(b_eq) if b_eq else None

        bounds = [(-move_limit, move_limit) for _ in range(n_vars)]

        res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')

        if not res.success:
            break

        delta_x = res.x
        x_next = x_current + delta_x
        f_next = target_func(x_next)

        current_eq_vals = [h(x_next) for h in eq_constraints]
        current_ineq_vals = [g(x_next) for g in ineq_constraints]

        max_violation = 0
        for val in current_eq_vals:
            max_violation = max(max_violation, abs(val))
        for val in current_ineq_vals:
            max_violation = max(max_violation, val)

        norm_delta_x_sq = np.dot(delta_x, delta_x)

        if norm_delta_x_sq <= epsilon and max_violation <= epsilon:
            history_data.append([q, x_next, f_next, current_eq_vals, current_ineq_vals])
            break

        x_current = x_next
        history_data.append([q, x_next, f_next, current_eq_vals, current_ineq_vals])

    return pd.DataFrame(history_data, columns=['iterasi', 'x', 'f(x)', 'h(x)', 'g(x)'])

In [ ]:
x0 = [0.5,0.5]
eq_constraints=[eq1]
ineq_constraints=[ineq1]
Ns=20
epsilon = 1e-6

sequential_linear_programming(objective_function, x0, eq_constraints, ineq_constraints, Ns, epsilon, 2.0)

,iterasi,x,f(x),h(x),g(x)
0,0,"[0.5, 0.5]",3.187500,[-1.5],[-0.75]
1,1,"[2.5, -1.4999861069231457e-05]",44.312687,[4.250000000224996],[0.5625000001687468]
2,2,"[1.6500096998278413, 1.9999850001389308]",6.544587,[4.7224720103066815],[2.6805880029670304]
3,3,"[1.1280349533432559, 1.249996250026319]",3.216951,[0.8349534810439816],[0.4899836828009254]
4,4,"[1.0072666883758488, 1.0250002249948167]",3.007796,[0.06521164275107427],[0.04161564130748108]
5,5,"[1.000026247847748, 1.0003050039338148]",3.000026,[0.0006625972794749302],[0.0004706997673833424]
6,6,"[1.0000000004756993, 1.0000000480238338]",3.000000,[9.699906833304794e-08],[7.227360199202337e-08]


# SQP

In [ ]:
def sequential_quadratic_programming(target_func, x0, eq_constraints, ineq_constraints, Ns, epsilon):

    def get_gradient(func, x):
        h_step = 1e-5
        n = len(x)
        grad = np.zeros(n)
        f_x = func(x)
        for i in range(n):
            x_tmp = x.copy()
            x_tmp[i] += h_step
            grad[i] = (func(x_tmp) - f_x) / h_step
        return grad

    def get_hessian(func, x):
        h_step = 1e-4
        n = len(x)
        hessian = np.zeros((n, n))
        grad_x = get_gradient(func, x)

        for i in range(n):
            x_tmp = x.copy()
            x_tmp[i] += h_step
            grad_tmp = get_gradient(func, x_tmp)
            hessian[:, i] = (grad_tmp - grad_x) / h_step

        return 0.5 * (hessian + hessian.T)

    x_current = np.array(x0, dtype=float)
    n_vars = len(x_current)

    f_val = target_func(x_current)
    val_eq_init = [h(x_current) for h in eq_constraints]
    val_ineq_init = [g(x_current) for g in ineq_constraints]

    history_data = []
    history_data.append([0, x_current.copy(), f_val, val_eq_init, val_ineq_init])

    for q in range(1, Ns + 1):
        grad_f = get_gradient(target_func, x_current)
        H_matrix = get_hessian(target_func, x_current)

        def qp_objective(d):
            return np.dot(grad_f, d) + 0.5 * np.dot(d, np.dot(H_matrix, d))

        cons = []

        for h_func in eq_constraints:
            grad_h = get_gradient(h_func, x_current)
            val_h = h_func(x_current)
            cons.append({'type': 'eq', 'fun': lambda d, gh=grad_h, vh=val_h: np.dot(gh, d) + vh})

        for g_func in ineq_constraints:
            grad_g = get_gradient(g_func, x_current)
            val_g = g_func(x_current)
            cons.append({'type': 'ineq', 'fun': lambda d, gg=grad_g, vg=val_g: -(np.dot(gg, d) + vg)})

        d0 = np.zeros(n_vars)

        res = minimize(qp_objective, d0, constraints=cons, method='SLSQP', tol=1e-8)

        delta_x = res.x
        x_next = x_current + delta_x
        f_next = target_func(x_next)

        current_eq_vals = [h(x_next) for h in eq_constraints]
        current_ineq_vals = [g(x_next) for g in ineq_constraints]

        norm_delta_x_sq = np.dot(delta_x, delta_x)

        max_violation = 0
        for val in current_eq_vals:
            max_violation = max(max_violation, abs(val))
        for val in current_ineq_vals:
            max_violation = max(max_violation, val)

        history_data.append([q, x_next, f_next, current_eq_vals, current_ineq_vals])

        if norm_delta_x_sq <= epsilon and max_violation <= epsilon:
            break

        x_current = x_next

    return pd.DataFrame(history_data, columns=['iterasi', 'x', 'f(x)', 'h(x)', 'g(x)'])

In [ ]:
x0 = [0.5,0.5]
eq_constraints=[eq1]
ineq_constraints=[ineq1]
Ns=20
epsilon = 1e-6

sequential_quadratic_programming(objective_function, x0, eq_constraints, ineq_constraints, Ns, epsilon)

,iterasi,x,f(x),h(x),g(x)
0,0,"[0.5, 0.5]",3.187500,[-1.5],[-0.75]
1,1,"[1.2499925076032277, 1.2499924938174547]",3.550754,[1.1249625036641846],[0.5624812432160358]
2,2,"[1.0249995513921288, 1.0249995487099488]",3.027547,[0.10124815520966379],[0.050624076230215254]
3,3,"[1.0003049877049073, 1.0003049876375818]",3.000305,[0.0012201367199375923],[0.0006100683262957318]
4,4,"[1.0000000480187952, 1.0000000480187856]",3.000000,[1.920751659945097e-07],[9.603757833431814e-08]
